<a href="https://colab.research.google.com/github/leongriner-portfolio/synth-haem/blob/main/synth_haem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

# Environment Setup (Run on Session Start)

As the simulation and analyses scripts need to run in the Google Colab environment, a number of resource constraints and workflows needs to be addressed.

The main issue that needs to be addressed is sessions are wiped periodically, so the environment needs to built/loaded each time. Due to space constraints, and large intermediate file sizes, we need to write to Google Drive periodically. While it would be possible to just write to local storage, it's faster and safer to just retrieve environment, reference, simulation and data files from the PROJECT (Google Drive) storage folder.

However, it's important to check that environment settings are all account for, directory structure is consistent etc.

The Environment Setup block of cells will do the following:
*   Specify settings, creating consistent directory structure and loads data, and load this into environment
*   Mount Google Drive storage and checks mounting
*   Copy stored data, reference files, etc into local
*   Sets tools in environment.yml, installs and builds environment
*   Checks versions, sets lock file and caches that environment so that it's consistent across runs/re-loads after session wipes
*   Saves local (WORK) files into Drive Storage (PROJECT)  

While these steps to maintain consistency, many are time-consuming and there are opportunites for error:
*   Accidently running the Environment Setup block of cells would overwrite local files that had been generated, if older version are in storage
*   Building the environment is time consuming and there are opportunities for version changes in between builds, which might result in unexpected behaviour/create issues with reproducibility

Consequently, re-running the block, after it's been run once effectively does nothing:
*   Mounting does nothing if already mounted
*   Copying Drive files does nothing unless a FORCE_RECOPY switch is set to TRUE (as restoring stored files if wanted is desirable behaviour).
*   Writing environment.yml does nothing if it already exists.
*   Instead of rebuilding from environment.yml, loads the cache, failing that uses lockfile. However, once cache is already generated, uses that (unless make_cache switch set to True)



## Settings, Load Data from Drive, Check Machine config

In [23]:
import os

# Create SETTINGS dictionary (dict)
# Reminder how dicts work here: {} used to set values, and subsequent [] used to ref
# First pass sets keys, then subsequent SETTINGS[key] = f{}

SETTINGS = {
    #Google drive folder
    "PROJECT": "/content/drive/MyDrive/Colab Notebooks/Project Storage/Bioinformatics Portfolio",
    #local dir
    "WORK": "/content/my_project",
    #locally-installed tools
    "TOOLS": "/content/tools",
    # Copy markers: recording when data last moved between Drive (PROJECT) and Local (WORK).
    # Deliberately OUTSIDE WORK, so saving WORK to Drive never carries them up.
    # Lost at wipe so new session will start with fresh copy markers.
    "MARKERS_DIR": "/content/markers",
    #mamba envir - note do not rename, name not ad hoc used by micromamba
    #unless remapping with os.environ["MAMBA_ROOT_PREFIX"] = SETTINGS[WHATEVER]
    "MAMBA_ROOT_PREFIX": "/content/micromamba",
    #project environment name
    "ENV_NAME": "bioinf_portfolio",
    #BAMsurgeon commit this was checked against - don't want it to break from upstream changes
    "BAMSURGEON_COMMIT": "d9448a5"
    }

# Derived paths, built from the settings above.
SETTINGS["ENV_DIR"] = f"{SETTINGS['PROJECT']}/env"                       # env files on Drive
SETTINGS["ENV_YML"] = f"{SETTINGS['ENV_DIR']}/environment.yml"           # what to install
SETTINGS["ENV_LOCK"] = f"{SETTINGS['ENV_DIR']}/bioinf_portfolio.lock.yml"      # exact versions written after the first build
SETTINGS["ENV_CACHE"] = f"{SETTINGS['ENV_DIR']}/bioinf_portfolio-env.tar.gz"   # optional packed copy of the built environment
SETTINGS["MICROMAMBA"] = f"{SETTINGS['TOOLS']}/bin/micromamba"                      # the micromamba program
SETTINGS["ENV_BIN"] = f"{SETTINGS['MAMBA_ROOT_PREFIX']}/envs/{SETTINGS['ENV_NAME']}/bin"  # the environment's tools
SETTINGS["LOGS_DIR"] = f"{SETTINGS['WORK']}/output/logs"                 # logs, local; later section saves them to Drive
SETTINGS["BAMSURGEON_DIR"] = f"{SETTINGS['TOOLS']}/bamsurgeon"                      # BAMSurgeon's code (re-cloned each session)
#note that version of BAMsurgeon that uses the spike-in is not packaged yet so code needs to be run
SETTINGS["DRIVE_COPY_MARKER"] = f"{SETTINGS['MARKERS_DIR']}/drive_to_work.txt"      # when Drive was last copied to WORK (section 2)
SETTINGS["WORK_COPY_MARKER"] = f"{SETTINGS['MARKERS_DIR']}/work_to_drive.txt"       # when WORK was last saved to Drive (save cells, later)


#export SETTINGS as environment variables
os.environ.update(SETTINGS)

# Keys later sections rely on, written out separately from SETTINGS so that a
# line missing from SETTINGS is caught here, not as an error sections later.
# Add a key here whenever a new section starts using one.
REQUIRED = [
    # Drive <-> WORK copying and saving sections
    "PROJECT", "WORK", "MARKERS_DIR", "DRIVE_COPY_MARKER", "WORK_COPY_MARKER",
    # micromamba and the environment sections
    "TOOLS", "MICROMAMBA", "MAMBA_ROOT_PREFIX", "ENV_NAME", "ENV_DIR",
    "ENV_YML", "ENV_LOCK", "ENV_CACHE", "ENV_BIN",
    # BAMSurgeon section
    "BAMSURGEON_DIR", "BAMSURGEON_COMMIT",
    # Logs section
    "LOGS_DIR",
]

# Checks if items in REQUIRED is missing from the environment.
missing = [key for key in REQUIRED if key not in os.environ]
if missing:
    raise RuntimeError(f"Missing from the environment: {missing}. Check section 1's SETTINGS.")
print(f"All {len(REQUIRED)} required settings exported.")


All 17 required settings exported.


NOTE: ONLY RUN AT START

Project will write to local disk, as it is faster, but will store to Drive to keep outputs between wipes.

In [2]:
import os
from pathlib import Path
import shutil
import time
from datetime import datetime, timezone
from google.colab import drive

# Mount Drive. If it is already mounted this session, this does nothing
# because force_remount defaults to False. To force remount call
# drive.mount("/content/drive", force_remount=True).
drive.mount("/content/drive")

#Check if mounted to prevent create issue writing fake dir structure
if not os.path.ismount("/content/drive"):
    raise RuntimeError("Google Drive is not mounted. Rerun this cell.")

Path(SETTINGS["PROJECT"]).is_dir() #checks Project exists in drive.

# Paths come from section 1: Setup - Environment, so they can't change
# Not explained in Section 1 but writing to local for speed reasons but using
# Google Drive as Project path for long term storage between Colab wipes
# This creates an issue where running this cell accidently would trigger re-copy
# This would over-write the existing WORK directory with old data.
# Copy-handling via FORCE_RECOPY and Copy Markers (DRIVE- and WORK-COPY-MARKER)
# Only change FORCE_RECOPY = True if over-writing data

FORCE_RECOPY = False

PROJECT_path = Path(SETTINGS["PROJECT"])
WORK_path = Path(SETTINGS["WORK"])
DRIVE_COPY_MARKER_path = Path(SETTINGS["DRIVE_COPY_MARKER"])  # written below, after each PROJECT -> WORK copy
WORK_COPY_MARKER_path = Path(SETTINGS["WORK_COPY_MARKER"])    # written by the save cells, after each WORK -> Drive copy

Path(SETTINGS["MARKERS_DIR"]).mkdir(parents=True, exist_ok=True)

# COPY HANDLING:
# Define the creating markers, used when stamping/reading copies from PROJECT and WORK
def stamp_marker(marker):
  """Write current UTC time into marker file for read_marker to read it back
  The save cells in later sections call this too, for WORK_COPY_MARKER_path.
  """
  marker.write_text(datetime.now(timezone.utc).isoformat(timespec="seconds"))

def read_marker(marker):
    """Return Time UTC, N min ago from a marker, or None if there's no marker yet."""
    if not marker.exists():
        return None
    when = datetime.fromisoformat(marker.read_text().strip())
    minutes = (datetime.now(timezone.utc) - when).total_seconds() / 60
    return f"{when:%Y-%m-%d %H:%M} UTC, {minutes:.0f} min ago"

#Check if a fresh copy or if FORCE_RECOPY = TRUE, otherwise don't copy PROJECT files.
if DRIVE_COPY_MARKER_path.exists() and not FORCE_RECOPY:
    # Not a fresh session: WORK may hold changes not yet saved to Drive, so don't copy.
    print("Did not copy from Drive: this session has already loaded the project.")
    print(f"  Last loaded from Drive:  {read_marker(DRIVE_COPY_MARKER_path)}")
    # 'or' gives 'not yet saved' when read_marker returns None (no save yet).
    # Reminder to self on how this works: 'None' returned from read_marker counts as
    # False (and string UTC date = True) when evaluating the f"'A' or 'B'.""
    print(f"  Last saved to Drive:     {read_marker(WORK_COPY_MARKER_path) or 'not saved yet'}")
    print("To copy anyway, set FORCE_RECOPY = True and re-run this cell.")
    print("A forced copy replaces local files with their Drive versions.")
else:
    # Fresh session (no Drive marker yet), or FORCE_RECOPY = True.
    start = time.time()
    # dirs_exist_ok=True lets copytree copy into WORK when it already exists.
    # Files in both places get the Drive version; files only in WORK are left alone.
    # ignore skips env/ (any file or folder named exactly env): its files are used
    # straight from Drive (section 1), and keeping them out of WORK means section 11
    # can never overwrite them with stale copies. It also skips *.partial files:
    # leftovers from an interrupted save (section 11).
    shutil.copytree(PROJECT_path, WORK_path, dirs_exist_ok=True, ignore=shutil.ignore_patterns("env", "*.partial"))
    # Stamp the marker only after the copy succeeds, so a failed copy runs again next time.
    stamp_marker(DRIVE_COPY_MARKER_path)
    print(f"Copied {PROJECT_path} -> {WORK_path} in {time.time() - start:.0f} s")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied /content/drive/MyDrive/Colab Notebooks/Project Storage/Bioinformatics Portfolio -> /content/my_project in 115 s


In [3]:
%%bash
# Check Machine:
# nproc: number of CPUs available (sets how many threads to use later. Expecting 2).
echo "CPUs: $(nproc)"
# free -h: memory human-readable. bwa index needs about 5 GB.
free -h
# df -h: free space on the local disk (where the working files go).
df -h /content
# and Drive (Reminder: || in bash works to run fallback if non-zero exit code)
df -h /content/drive 2>/dev/null || echo "Drive not mounted yet: run section 2 first."

CPUs: 2
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.5Gi       246Mi       3.0Mi        11Gi        11Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   19G   89G  18% /
Filesystem      Size  Used Avail Use% Mounted on
drive           100G   89G   12G  89% /content/drive


## Dirs and Env, Micromamba install, load environment

Creates the folders this workflow needs, including env/ on Drive, then writes environment.yml there (only if it isn't there yet).

It lists every package, plus r-base for the results table and pip/setuptools for installing BAMSurgeon.

This lives on Drive.

**Reminder to self:**
In Colab, each %%bash cell starts a fresh shell.
Without pipefail, default behaviour is to continue past errors, treat unset variables as empty text, and checks pipeline if last command worked.

Conversely:

```
set -e            # stop at the first failing command
set -u            # treat an unset variable as an error
set -o pipefail   # a pipeline fails if any command in it fails
```

**BAMsurgeon:** tools for adding mutations to existing .bam files, used for testing mutation callers. Importantly, an August 2026 commit allows for spike for low VAF mutations.

Bioconda’s only build is noarch/bamsurgeon-1.4.1-pyhdfd78af_0.tar.bz2, uploaded Jun 13, 2024. It packages the GitHub release from August 24, 2022, described as a bugfix release for 1.4, read pairs were not maintained in spike-in.

Section 1 pins GitHub commit d9448a5, from August 2026. That code has never been released as a new version, and its pyproject.toml still says version = "1.4". So the newer code carries the smaller number. This commit adds the features the spike-in step relies

In [ ]:
%%bash
set -euo pipefail

#-p doesn't complain if parent exists
# $TOOLS/bin and MAMBA_ROOT_PREFIX are local
# ENV_DIR is on Drive

mkdir -p "$TOOLS/bin" "$MAMBA_ROOT_PREFIX" "$ENV_DIR"

#write environment.yml only if not on drive (i.e., should only do something on first run)
if [ ! -f "$ENV_YML" ]; then #if not a file...
 cat > "$ENV_YML" <<'EOF'
# Software for the synthetic haem panel dataset.
# Channel order matters: conda-forge first, then bioconda, as bioconda expects.
name: bioinf_portfolio
channels:
  - conda-forge
  - bioconda
dependencies:
  - python=3.11      # pinned so BAMSurgeon's pip install has a known base
  - pip              # for installing BAMSurgeon from its GitHub source
  - setuptools=84.2  # tested with BAMSurgeon d9448a5; newer versions may reject its licence format after Feb 2027
  - samtools         # index FASTA; sort, index, subsample, merge BAMs; BAM to FASTQ
  - htslib           # bgzip and tabix
  - bcftools         # sort, normalise, filter, compare, query VCFs
  - bedtools         # pad, merge, intersect, extract regions
  - bwa              # index the reference; align reads
  - art              # ART read simulator (art_illumina)
  - wgsim            # BAMSurgeon dependency (its SV engine)
  - pysam            # BAMSurgeon dependency
  - mosdepth         # depth over targets
  - picard           # interval lists; capture metrics
  - gatk4            # dictionary, MarkDuplicates, Mutect2, FilterMutectCalls, HaplotypeCaller
  - rtg-tools        # reference SDF; vcfeval scoring
  - jq               # pull fields out of JSON replies (Variant Recoder)
  - curl             # downloads and web API queries
  - r-base           # results table (base R only)
EOF
  echo "Wrote $ENV_YML (on Drive, so later sessions keep it)."
else
  echo "Using environment.yml from Drive (edit it there; delete it to regenerate)."
fi

# Show what will be installed.
cat "$ENV_YML"

Using environment.yml from Drive (edit it there; delete it to regenerate).
# Software for the synthetic haem panel dataset.
# Channel order matters: conda-forge first, then bioconda, as bioconda expects.
name: bioinf_portfolio
channels:
  - conda-forge
  - bioconda
dependencies:
  - python=3.11      # pinned so BAMSurgeon's pip install has a known base
  - pip              # for installing BAMSurgeon from its GitHub source
  - setuptools       # BAMSurgeon's manual asks for setuptools >= 61
  - samtools         # index FASTA; sort, index, subsample, merge BAMs; BAM to FASTQ
  - htslib           # bgzip and tabix
  - bcftools         # sort, normalise, filter, compare, query VCFs
  - bedtools         # pad, merge, intersect, extract regions
  - bwa              # index the reference; align reads
  - art              # ART read simulator (art_illumina)
  - wgsim            # BAMSurgeon dependency (its SV engine)
  - pysam            # BAMSurgeon dependency
  - mosdepth         # depth ov

In [ ]:
%%bash
set -euo pipefail

# skip if already has it this session.
if [ ! -x "$MICROMAMBA" ]; then
  cd "$TOOLS"
  # curl options: -L follow redirects, -s silent. The download is a .tar.bz2
  # archive; tar -x extracts, -v lists files, -j means bzip2, and
  # bin/micromamba extracts only that one file (to $TOOLS/bin/micromamba).
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
fi

# Print the version to confirm it runs.
"$MICROMAMBA" --version

bin/micromamba
2.10.0


## Build/restore environment

First pass: build from .yml. Later passes build from either lock file or unpack cached .tar.gz environment stored in Drive.

If version issues are occurring, change the versions in the environment.yml and rebuild from there (deleting the lock and cache from Drive first).

To specify version, change the generic package name:

In environment.yml | Matches
-------------------|------------------
samtools=1.20 | any 1.20.x release (1.20, 1.20.1, …) |
samtools==1.20 |	exactly 1.20
samtools>=1.20,<1.22 | a range
samtools=1.20=h50ea8bc_0 | one exact build (rarely needed; this one made up)

To see which versions exist before pinning:
`"$MICROMAMBA" search -c conda-forge -c bioconda samtools`




In [4]:
%%bash
set -euo pipefail

ENV_PATH="$MAMBA_ROOT_PREFIX/envs/$ENV_NAME"
# SECONDS is a bash built-in counter, used here to time the step.
SECONDS=0

if [ -d "$ENV_PATH" ]; then
  echo "Environment already present in this session: $ENV_PATH"

elif [ -f "$ENV_CACHE" ]; then
  echo "Restoring the cached environment..."
  # Unpack into MAMBA_ROOT_PREFIX's parent folder (/content), where we'll later pack
  # it from. Conda environments contain hard-coded paths, so they only work at
  # the path they were built in. -x extract, -z gzip, -f the archive file.
  tar -C "$(dirname "$MAMBA_ROOT_PREFIX")" -xzf "$ENV_CACHE"

elif [ -f "$ENV_LOCK" ]; then
  echo "Rebuilding from the lock file (exact versions from the first build)..."
  # create: make a new environment. -n: its name. -f: the file listing
  # packages. -y: answer yes to the confirmation prompt. micromamba puts it
  # under MAMBA_ROOT_PREFIX, set in Cell 1: Settings.
  "$MICROMAMBA" create -y -n "$ENV_NAME" -f "$ENV_LOCK"

else
  echo "First build from environment.yml (latest versions)..."
  "$MICROMAMBA" create -y -n "$ENV_NAME" -f "$ENV_YML"
fi

# Check the environment landed where ENV_BIN) expects it.
# -x: the file exists and can be run.
if [ ! -x "$ENV_BIN/python3" ]; then
  echo "ERROR: no environment at $ENV_PATH."
  echo "Check that section 1 ran in this session and sets MAMBA_ROOT_PREFIX."
  exit 1
fi

echo "Done in $((SECONDS / 60)) min $((SECONDS % 60)) s."

Restoring the cached environment...
Done in 0 min 51 s.


**Make PATH variable:**

Made after tools are in environment's bin, following environment being built/loaded.

In [20]:
import os
import shutil

env_bin = os.environ["ENV_BIN"]

# Prepend the env's bin folder to PATH (only once, even if this cell is re-run).
if not os.environ["PATH"].startswith(env_bin):
    os.environ["PATH"] = env_bin + ":" + os.environ["PATH"]

# Check that the tool resolves to environment's bin folder. Uses
for tool in ["python3", "samtools", "gatk"]:
   found = shutil.which(tool)
   if found and found.startswith(env_bin + "/"): # "/" at end to avoid statswith being fooled
      print(f"Confirmed: {tool} -> {found}")
   else:
      raise RuntimeError(f"{tool} resolves to {found}, not the environment.")


Confirmed: python3 -> /content/micromamba/envs/bioinf_portfolio/bin/python3
Confirmed: samtools -> /content/micromamba/envs/bioinf_portfolio/bin/samtools
Confirmed: gatk -> /content/micromamba/envs/bioinf_portfolio/bin/gatk


Installing BAMsurgeon (if required) - takes ~30s

In [6]:
%%bash
set -euo pipefail

# Clone only if this session doesn't have it yet. Remove --quiet if needing progress
if [ ! -d "$BAMSURGEON_DIR/.git" ]; then
  git clone --quiet https://github.com/adamewing/bamsurgeon.git "$BAMSURGEON_DIR"
fi
cd "$BAMSURGEON_DIR"

# Switch to the exact commit the doc was checked against.
git checkout --quiet "$BAMSURGEON_COMMIT"
echo "BAMSurgeon at commit: $(git rev-parse --short HEAD)"

# ---- Install BAMSurgeon into the environment -----------------------------
#
# Why there's a build step at all:
#   BAMSurgeon arrives as source code, not a ready-made package, so pip has to
#   build it before installing it. pip doesn't do the build itself: it hands
#   the job to the build tool the project names in its pyproject.toml:
#       [build-system]
#       requires = ["setuptools>=61"]
#   i.e. "build me with setuptools, version 61 or newer".
#
# What pip does by default ("build isolation"):
#   It makes a throwaway environment (the /tmp/pip-build-env-... path in the
#   old warnings), downloads the NEWEST setuptools from PyPI into it, builds
#   there, then deletes it. Our environment's own setuptools is ignored.
#   Two problems with that:
#   1. The build tool can change between sessions without us knowing, and the
#      version used is never recorded (it isn't in the lock file).
#   2. Recent setuptools warns that BAMSurgeon's pyproject.toml declares its
#      licence in an outdated format (license = { file = "LICENSE.txt" }
#      instead of a short identifier such as license = "MIT"), and says
#      support for it ends after 2027-02-18. A setuptools released after that
#      date could refuse to build BAMSurgeon, and this cell would start
#      failing with nothing changed on our side.
#
# The fix, --no-build-isolation:
#   pip skips the throwaway environment and builds with the setuptools already
#   installed in OUR environment: the one micromamba installed from
#   environment.yml and the lock file records (84.2.1 in the current build).
#   Same build tool every session, nothing extra downloaded, and no future
#   setuptools release can affect it.
#   The catch: pip no longer fetches build tools for us, so setuptools must
#   already be in the environment. It is, because environment.yml lists it.
#   (If it ever went missing, this line would fail with an error about
#   setuptools or the "build backend".)
#   You may still see the licence warning: setuptools 84 prints it too. It is
#   only a warning, and with this setuptools version fixed it can't become an
#   error.
#
# Why the old 'pip install --upgrade pip setuptools' line is gone:
#   The environment already has pip and setuptools far newer than BAMSurgeon
#   needs. Upgrading them with pip would replace the copies micromamba
#   installed, so the environment would no longer match the lock file.
#
# The rest of the command:
#   python3 -m pip   run the pip belonging to the environment's Python (first
#                    on PATH since section 7), not Colab's own pip
#   --quiet          hide progress output; errors and warnings still show
#   -e               "editable": installs the bamsurgeon command, but it runs
#                    the code in this folder rather than a copy, as the
#                    manual's install instructions do
#   .                what to install: the current folder ($BAMSURGEON_DIR, from
#                    the cd above), which holds pyproject.toml
python3 -m pip install --quiet --no-build-isolation -e .

# BAMSurgeon's own check for samtools, bwa, wgsim and pysam.
# -O is how its README runs the script (it only disables Python assert checks).
python3 -O scripts/check_dependencies.py

BAMSurgeon at commit: d9448a5
All dependencies are satisfied


  /content/micromamba/envs/bioinf_portfolio/lib/python3.11/site-packages/setuptools/config/_apply_pyprojecttoml.py:82: BuildBackendWarning: `project.license` as a TOML table is deprecated
  !!
  
          ********************************************************************************
          Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).
  
          By 2027-Feb-18, you need to update your project and remove deprecated calls
          or your builds will no longer be supported.
  
          See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
          ********************************************************************************
  
  !!
    corresp(dist, value, root_dir)
  /content/micromamba/envs/bioinf_portfolio/lib/python3.11/site-packages/setuptools/config/_apply_pyprojecttoml.py:82: BuildBackendWarning: `project.

## Check versions, Write lockfile, Cache environment

In [7]:
%%bash
# No 'set -e' here: several tools exit with an error code when asked for help
# or a version, and that shouldn't stop the cell.
set -uo pipefail

# v LABEL COMMAND: print a heading, then the first 3 lines of the command's
# output (stderr included, since some tools print versions there).
v() {
  local label="$1"; shift
  echo "== $label"
  { bash -c "$*" 2>&1 || true; } | head -n 3
}

# The version list goes in WORK's output/logs (LOGS_DIR, section 1), like every
# other log. Section 11 saves it to Drive.
mkdir -p "$LOGS_DIR"
VERSIONS="$LOGS_DIR/tool_versions.txt"
{
  echo "Recorded: $(date -u '+%Y-%m-%d %H:%M UTC')"
  v "python"     "python3 --version"
  v "micromamba" "\"$MICROMAMBA\" --version"
  v "samtools"   "samtools --version"
  v "htslib"     "bgzip --version"
  v "bcftools"   "bcftools --version"
  v "bedtools"   "bedtools --version"
  v "bwa"        "bwa 2>&1 | grep -i '^version'"            # bwa prints its version in the usage text
  v "art"        "art_illumina 2>&1 | grep -i -m1 'version'" # same for ART
  v "wgsim"      "wgsim 2>&1 | grep -i -m1 'version'"
  v "mosdepth"   "mosdepth --version"
  v "picard"     "picard SortSam --version"
  v "gatk"       "gatk --version 2>&1 | grep -m1 'Genome Analysis Toolkit'"   # skip the launcher's preamble
  v "rtg-tools"  "rtg version"
  v "java"       "java -version"
  v "jq"         "jq --version"
  v "curl"       "curl --version"
  v "R"          "R --version"
  v "bamsurgeon commit" "git -C \"$BAMSURGEON_DIR\" rev-parse --short HEAD"   # the exact code installed
} | tee "$VERSIONS"   # tee prints to the screen AND writes the file
echo "Versions written to $VERSIONS (section 11 saves them to Drive)."

# Fail loudly if any command is missing from PATH.
missing=()
for tool in python3 samtools bgzip tabix bcftools bedtools bwa art_illumina wgsim \
            mosdepth picard gatk rtg java jq curl Rscript bamsurgeon; do
  command -v "$tool" > /dev/null || missing+=("$tool")
done
if [ ${#missing[@]} -gt 0 ]; then
  echo "MISSING: ${missing[*]}"
  exit 1
fi
echo "All tools found."

# Write the lock file once, after the first successful build. ENV_LOCK is in
# env/ on Drive (section 1), so it's saved there directly.
if [ ! -f "$ENV_LOCK" ]; then
  # env export lists every installed package with its exact version and build.
  "$MICROMAMBA" env export -n "$ENV_NAME" > "$ENV_LOCK"
  echo "Wrote the lock file to Drive: $ENV_LOCK"
fi

Recorded: 2026-11-22 22:41 UTC
== python
Python 3.11.18
== micromamba
bash: line 1: /content/tools/bin/micromamba: No such file or directory
== samtools
samtools 1.24
Using htslib 1.24
Copyright (C) 2026 Genome Research Ltd.
== htslib
bgzip (htslib) 1.24
Copyright (C) 2026 Genome Research Ltd.
== bcftools
bcftools 1.24
Using htslib 1.24
Copyright (C) 2026 Genome Research Ltd.
== bedtools
bedtools v2.31.1
== bwa
Version: 0.7.19-r1273
== art
          Q Version 2.5.8 (June 6, 2016)       
== wgsim
Version: 1.24
== mosdepth
mosdepth 0.3.14
== picard
Version:3.5.0
== gatk
The Genome Analysis Toolkit (GATK) v4.6.2.0
== rtg-tools
Product: RTG Tools 3.13
Core Version: c27844a5bb (2025-05-27)
RAM: 11.4GB of 12.6GB RAM can be used by rtg (90%)
== java
openjdk version "17.0.18-internal" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18-internal+0-adhoc.rattler.src)
OpenJDK 64-Bit Server VM (build 17.0.18-internal+0-adhoc.rattler.src, mixed mode, sharing)
== jq
jq-1.8.2
== curl
curl 8.23.0 (x

### OPTIONAL: Cache Environment

Takes ~3 mintes

In [ ]:
%%bash
set -euo pipefail

# Failsafe: only pack the cache when this is true. Set it back to false
# afterwards, so "Run all" never re-packs it by accident. (Bash has no
# True/False: the convention is the plain words true and false.)
MAKE_CACHE=false

if [ "$MAKE_CACHE" != true ]; then
  echo "Cache not written: MAKE_CACHE is false. Set it to true and re-run to pack the environment."
  exit 0    # end the cell here, successfully
fi

# Split MAMBA_ROOT_PREFIX (/content/micromamba) into its parent (/content) and its
# own name (micromamba), so the archive holds paths section 6 can unpack to
# the same place.
PARENT="$(dirname "$MAMBA_ROOT_PREFIX")"
NAME="$(basename "$MAMBA_ROOT_PREFIX")"

# ENV_CACHE is in env/ on Drive (section 1), so this writes straight there.
# It writes under a temporary name (.partial) and renames only once tar has
# finished. If the write is interrupted, only the .partial file is left:
# section 6 looks for the exact cache name, so it ignores it and falls back to
# the old cache or the lock file. Writing straight to the real name would wipe
# the old cache at the start and could leave a half-written one for section 6.
#   -C "$PARENT"          pack paths relative to the parent folder
#   --exclude="$NAME/pkgs" skip micromamba's download cache, not needed to run
#   -c create   -z gzip   -f output file
tar -C "$PARENT" --exclude="$NAME/pkgs" -czf "$ENV_CACHE.partial" "$NAME"
# set -e (top of the cell) means this only runs if tar succeeded.
mv "$ENV_CACHE.partial" "$ENV_CACHE"

# -h: human-readable size.
ls -lh "$ENV_CACHE"

Cache not written: MAKE_CACHE is false. Set it to true and re-run to pack the environment.


# Save WORK (Local) to PROJECT (Drive) storage

Run this whenever there's work worth saving (before the session ends): anything only in WORK is lost then. It isn't part of the start-of-session run.

Notes on behaviour:

*   Off by default. Set SAVE_TO_DRIVE = True to save, and back to False afterwards. When it's off, the cell only reports when you last saved.
*   Copies only what changed. Files whose Drive copy has the same size and modification time are skipped, so large BAMs aren't re-copied on every save. This relies on Drive keeping modification times; if it doesn't, every file is copied each time, which is slower but still correct. A second save straight after the first should report almost everything as unchanged.
*   Doesn't leave a half-written file on Drive. Each file is copied under a temporary .partial name and renamed only once it's complete, so if a copy fails, Drive keeps its previous version.
*   Never deletes anything on Drive, and skips env/ and .partial files.
*   Reports failures and stops. If anything can't be saved, it lists what (up to 20 items), leaves the save marker unchanged and stops with an error. Fix the cause and re-run: only what's still missing or changed gets copied.
*   Stops if Drive isn't mounted, so a save can't land in a fake Drive folder on the VM's own disk.

**If a save fails:**

Error says | Cause | Fix
-----------|----------|------
No space left on device or quota |  Drive is full | Free space (empty Drive's Bin too: deleted files still count), then re-run
Transport endpoint is not connected, or nearly every file fails | The Drive mount dropped | drive.mount("/content/drive", force_remount=True), then re-run
Input/output error, timeouts | A temporary Drive hiccup | Usually just re-run

In [ ]:
from posix import truncate
from pathlib import Path
import os
import shutil
import time

# Save WORK back to Drive (PROJECT). Run whenever there's work worth keeping,
# and always before the session ends: anything only in WORK is lost then.
# stamp_marker and read_marker come from section 2. After Runtime -> Restart
# session, re-run sections 1 and 2 first (section 2 won't copy again).

# Failsafe: only save when this is True. Set it back to False afterwards,
# so "Run all" never saves by accident.
SAVE_TO_DRIVE = False

PROJECT_path = Path(SETTINGS["PROJECT"])
WORK_path = Path(SETTINGS["WORK"])
WORK_COPY_MARKER_path = Path(SETTINGS["WORK_COPY_MARKER"])

# Tally of what the save did, filled in by copy_changed_safely below.
counts = {"copied": 0, "unchanged": 0}

def copy_changed_safely(src, dst):
    """Copy one file from WORK to Drive. copytree calls this for every file.

    1. Skip it if Drive already has an identical copy (same size and
       modification time), so unchanged files, large BAMs included, aren't
       re-copied on every save.
    2. Otherwise copy it to dst.partial and rename that over dst once it's
       complete, so a failed copy never leaves a half-written file under the
       real name: Drive keeps its previous version.
    """
    if os.path.exists(dst):
        s, d = os.stat(src), os.stat(dst)
        # copy2 keeps a file's modification time, so a file unchanged since it
        # was copied matches its Drive copy. The 2-second leeway allows for
        # Drive storing times less precisely. If Drive doesn't keep the times
        # at all, nothing matches and every file is copied: slower, still correct.
        if s.st_size == d.st_size and abs(s.st_mtime - d.st_mtime) < 2:
            counts["unchanged"] += 1
            return dst
    tmp = f"{dst}.partial"
    shutil.copy2(src, tmp)   # the slow part; if it fails, dst is untouched
    os.replace(tmp, dst)     # a rename: the real name never points at a half-written file
    counts["copied"] += 1
    return dst


if not SAVE_TO_DRIVE:
    print("Not saved: SAVE_TO_DRIVE is False. Set it to True and re-run this cell to save.")
    print(f"  Last saved to Drive:     {read_marker(WORK_COPY_MARKER_path) or 'not saved yet'}")
else:
    # Stop if Drive isn't mounted, so the copy can't land in a fake Drive folder
    # on the VM's own disk.
    if not os.path.ismount("/content/drive"):
        raise RuntimeError("Google Drive is not mounted. Run section 2 first.")

    start = time.time()
    try:
        # dirs_exist_ok=True copies into the existing Drive folder. Nothing on
        # Drive is deleted: a file you delete in WORK stays on Drive until you
        # delete it there too.
        # ignore skips env/ (any file or folder named exactly env): its files live
        # on Drive and are written there directly, so Drive's copy is always the
        # current one. It also skips *.partial files, so leftovers from an
        # interrupted save are never copied around.
        shutil.copytree(WORK_path, PROJECT_path, dirs_exist_ok=True,
                        copy_function=copy_changed_safely,
                        ignore=shutil.ignore_patterns("env", "*.partial"))
    except shutil.Error as e:
        # copytree keeps going past failures and reports them all at the end:
        # e.args[0] is a list of (source, destination, reason).
        failed = e.args[0]
        print(f"SAVE INCOMPLETE: {len(failed)} item(s) were not saved to Drive.")
        for src, dst, why in failed[:20]:   # first 20, so a dropped mount can't flood the output
            print(f"  {Path(src).relative_to(WORK_path)}: {why}")
        if len(failed) > 20:
            print(f"  ... and {len(failed) - 20} more")
        print(f"Everything else was saved ({counts['copied']} copied, {counts['unchanged']} unchanged).")
        print("Drive still has the previous version of each failed file, if it had one.")
        print("Fix the cause (see the table above this cell) and re-run: only what's still")
        print("missing or changed is copied. The save marker was NOT updated.")
        # Stop with a short error: the cell turns red and "Run all" stops here.
        # 'from None' hides the original exception's huge list.
        raise RuntimeError(f"Save incomplete: {len(failed)} item(s) failed") from None

    # Stamp the marker only after a complete save.
    stamp_marker(WORK_COPY_MARKER_path)
    print(f"Saved {WORK_path} -> {PROJECT_path} in {time.time() - start:.0f} s: "
          f"{counts['copied']} file(s) copied, {counts['unchanged']} unchanged.")
    print(f"  Last loaded from Drive:  {read_marker(Path(SETTINGS['DRIVE_COPY_MARKER']))}")
    print(f"  Last saved to Drive:     {read_marker(WORK_COPY_MARKER_path)}")

Saved /content/my_project -> /content/drive/MyDrive/Colab Notebooks/Project Storage/Bioinformatics Portfolio in 0 s: 1 file(s) copied, 0 unchanged.
  Last loaded from Drive:  2026-11-19 11:22 UTC, 196 min ago
  Last saved to Drive:     2026-11-19 14:38 UTC, 0 min ago


# Build Simulation Files

## SETUP: New Settings, check files

In [21]:
import os
import shutil
from pathlib import Path

# Needs notebook 01's sections 1-9 run first in this session: SETTINGS, the
# PATH change (section 7) and BAMSurgeon (section 8) all come from there.
if "SETTINGS" not in globals():
    raise RuntimeError("SETTINGS is missing: run notebook 01 sections 1-9 first.")

# Keep BAMs on Drive? False (default) puts them in local scratch, which section
# 11 never saves: they are about 1 GB each and can be rebuilt from the FASTQs.
# True puts them under WORK, so section 11 saves them. Check Drive space first.
KEEP_BAMS_ON_DRIVE = False

WORK = SETTINGS["WORK"]
NB2 = {
    # Local-only space for big temporary files. It sits outside WORK, so it is
    # never saved to Drive, and it disappears when the session ends.
    "SCRATCH": "/content/scratch",
    "REF_DIR": f"{WORK}/reference/genome",          # masked GRCh38 and its indexes (step 0)
    "DESIGN_DIR": f"{WORK}/reference/sureselect",   # Agilent files, cleaned BEDs (step 1)
    "TRUTH_DIR": f"{WORK}/data/raw/truth",          # spike-in list, truth VCFs (steps 2, 6)
    "SIM_DIR": f"{WORK}/data/raw/sim",              # read-source regions, FASTQs (steps 3, 4, 7)
    "CALLS_DIR": f"{WORK}/data/processed/calls",    # the test pipeline's VCFs (step 8)
    "QC_DIR": f"{WORK}/output/results/qc",          # depth, capture, spike-in reports
    "EVAL_DIR": f"{WORK}/output/results/eval",      # scoring and the results table (step 9)
}
NB2["BAM_DIR"] = f"{WORK}/data/processed/bam" if KEEP_BAMS_ON_DRIVE else f"{NB2['SCRATCH']}/bam"

# Files that several steps use.
NB2["REF"] = f"{NB2['REF_DIR']}/hg38_masked.fa"           # the reference FASTA
NB2["SDF"] = f"{NB2['REF_DIR']}/hg38_masked.sdf"          # RTG's copy of it (a folder)
NB2["GENOME"] = f"{NB2['REF_DIR']}/hg38.genome"           # contig lengths, for bedtools
NB2["HAEM_DIR"] = f"{NB2['DESIGN_DIR']}/Haem_1"           # the unzipped Agilent design
NB2["TARGETS_BED"] = f"{NB2['DESIGN_DIR']}/Regions_SNV.clean.bed"
NB2["BAITS_BED"] = f"{NB2['DESIGN_DIR']}/Covered_SNV.clean.bed"
NB2["TARGETS_IL"] = f"{NB2['DESIGN_DIR']}/targets.interval_list"
NB2["BAITS_IL"] = f"{NB2['DESIGN_DIR']}/baits.interval_list"

# CPUs this session has (2 on free Colab). Every multithreaded command uses
# this, so nothing asks for more threads than the machine has.
NB2["THREADS"] = str(os.cpu_count() or 2)
# Java memory for GATK and Picard: 4 GB of the ~12.7 GB, leaving room others.
# MarkDuplicates would probably use most but 1.2Mb + 8M reads should fit.
NB2["JAVA_MEM"] = "-Xmx4g"

# Keys the step cells read as $NAME. Environment Setup's setting are listed too,
# so it means issues are caught here, not hours later in a step.
REQUIRED_NB2 = [
    # From Environment Setup section
    "PROJECT", "WORK", "LOGS_DIR", "BAMSURGEON_DIR",
    # Folders
    "SCRATCH", "REF_DIR", "DESIGN_DIR", "TRUTH_DIR", "SIM_DIR",
    "CALLS_DIR", "QC_DIR", "EVAL_DIR", "BAM_DIR",
    # Files several steps use
    "REF", "SDF", "GENOME", "HAEM_DIR", "TARGETS_BED", "BAITS_BED",
    "TARGETS_IL", "BAITS_IL",
    # Resources
    "THREADS", "JAVA_MEM",
]
missing = [key for key in REQUIRED_NB2 if key not in os.environ]
if missing:
    raise RuntimeError(f"Missing from the environment: {missing}. "
                       "Re-run notebook 01 section 1, then this cell.")

# Export for the %%bash cells, then make every folder. exist_ok=True: no error
# if a folder is already there (for example, copied down from Drive).
os.environ.update(NB2)
for key in ("SCRATCH", "REF_DIR", "DESIGN_DIR", "TRUTH_DIR", "SIM_DIR",
            "CALLS_DIR", "QC_DIR", "EVAL_DIR", "BAM_DIR"):
    Path(NB2[key]).mkdir(parents=True, exist_ok=True)

# Check every tool this notebook calls can be found. shutil.which searches
# PATH the way the shell does, so this also confirms section 7 ran.
tools = ["samtools", "bcftools", "bedtools", "bgzip", "tabix", "bwa", "art_illumina",
         "mosdepth", "picard", "gatk", "rtg", "bamsurgeon", "Rscript", "curl"]
missing = [t for t in tools if shutil.which(t) is None]
if missing:
    raise RuntimeError(f"Not found on PATH: {missing}. Run notebook 01 sections 6-8 first.")

for key in NB2:
    print(f"{key:12} {os.environ.get(key, 'NOT SET')}")
print("All tools found.")

SCRATCH      /content/scratch
REF_DIR      /content/my_project/reference/genome
DESIGN_DIR   /content/my_project/reference/sureselect
TRUTH_DIR    /content/my_project/data/raw/truth
SIM_DIR      /content/my_project/data/raw/sim
CALLS_DIR    /content/my_project/data/processed/calls
QC_DIR       /content/my_project/output/results/qc
EVAL_DIR     /content/my_project/output/results/eval
BAM_DIR      /content/scratch/bam
REF          /content/my_project/reference/genome/hg38_masked.fa
SDF          /content/my_project/reference/genome/hg38_masked.sdf
GENOME       /content/my_project/reference/genome/hg38.genome
HAEM_DIR     /content/my_project/reference/sureselect/Haem_1
TARGETS_BED  /content/my_project/reference/sureselect/Regions_SNV.clean.bed
BAITS_BED    /content/my_project/reference/sureselect/Covered_SNV.clean.bed
TARGETS_IL   /content/my_project/reference/sureselect/targets.interval_list
BAITS_IL     /content/my_project/reference/sureselect/baits.interval_list
THREADS      2
JAVA_MEM 

## STEP 0: Get Index and Masked References

Using GIAB's masked GRCh38 (no alt contigs, and GRC's false duplication fix for chr21 copy of U2AF1). Unpack FASTA and generate .fai index.

Note: GIAB's masked GRCh38: NCBI's no-alt analysis set with the Genome Reference Consortium's false duplications replaced by Ns, including the chr21 copy of U2AF1 that Miller et al. described. Coordinates don't change, and GIAB maps its own data to this family of references, which keeps the later HG002 check consistent.
*   In: GIAB's GCA_000001405.15_GRCh38_no_alt_analysis_set_maskedGRC_exclusions_v2.fasta.gz, downloaded into scratch.
*   Out: reference/genome/hg38_masked.fa with its .fai, .dict, bwa index and RTG SDF, plus hg38.genome and the step0.done marker.
*   Then save: recommended to toggle the Save switch (Save Work cell) and run straight away. Losing this to a disconnect is costly.

In [24]:
%%bash
set -euo pipefail

# Rebuild switch: set to true only to build the reference again from scratch.
# Again, note no TRUE boolean in bash, this is just a string.
# Note: this will still build ref first time even if REDO = false.
REDO=false

# step0.done is written last, so it only exists after a complete build. Section
# 2 copies it down from Drive with the reference, so later sessions skip this.
DONE="$REF_DIR/step0.done"
if [ "$REDO" != true ] && [ -f "$DONE" ] && [ -s "$REF.bwt" ] && [ -d "$SDF" ]; then
  echo "Reference and indexes already in $REF_DIR: skipped."
  exit 0
fi
rm -f "$DONE"
SECONDS=0   # bash's built-in timer

# Drive needs about 10 GB free for the result (saved in section 11).
# -B1G reports in whole GB; --output=avail prints only the free space.
FREE_GB=$(df -B1G --output=avail /content/drive | tail -n 1 | tr -d ' ')
echo "Drive free space: ${FREE_GB} GB (the reference needs about 10 GB)."
if [ "$FREE_GB" -lt 11 ]; then
  echo "WARNING: free up Drive space before saving this step in section 11."
fi

# Download GIAB's masked GRCh38 (about 0.9 GB) into scratch: only the
# unpacked FASTA is kept. Why this file:
#   no_alt_analysis_set      primary chromosomes only, so reads don't get two
#                            equally good homes on alternate-haplotype contigs.
#   maskedGRC_exclusions_v2  regions the GRC flagged as false duplications are
#                            Ns, so reads from the real U2AF1 (chr21q) map uniquely.
#   chr-prefixed names       chr1, chr2 ... match the Agilent BED files.
# curl: -f fail on an HTTP error (so set -e stops the cell), -L follow
# redirects, --retry 3 retry a dropped connection, -o output file.
FA_GZ="$SCRATCH/GCA_000001405.15_GRCh38_no_alt_analysis_set_maskedGRC_exclusions_v2.fasta.gz"
curl -fL --retry 3 -o "$FA_GZ" \
  "https://ftp-trace.ncbi.nlm.nih.gov/ReferenceSamples/giab/release/references/GRCh38/$(basename "$FA_GZ")"

# Unpack to plain FASTA (support for compressed references varies between
# tools), then delete the download to free scratch space.
gunzip -c "$FA_GZ" > "$REF"
rm -f "$FA_GZ"

# .fai index: lets tools jump to any region without reading the whole 3 GB.
samtools faidx "$REF"

# Two-column genome file (contig, length): bedtools uses it in step 3 so
# padding stops at chromosome ends.
cut -f1,2 "$REF.fai" > "$GENOME"

# .dict sequence dictionary: GATK and Picard refuse a reference without one.
# ${REF%.fa}.dict means "REF with .fa cut off the end, plus .dict". GATK won't
# overwrite an old one, so remove it first.
rm -f "${REF%.fa}.dict"
gatk --java-options "$JAVA_MEM" CreateSequenceDictionary -R "$REF"

# bwa's search index: the slow part, about an hour (estimate) and about 5 GB
# RAM, single-threaded. Its progress messages go to a log file instead of
# flooding the cell.
echo "Building the bwa index (about an hour)..."
bwa index "$REF" 2> "$LOGS_DIR/step0_bwa_index.log"

# RTG's own copy of the reference (an SDF folder). vcfeval (step 9) compares
# variants by the sequence they produce, and needs it. rtg refuses an
# existing output folder, so remove any old one first.
rm -rf "$SDF"
rtg format -o "$SDF" "$REF" > "$LOGS_DIR/step0_rtg_format.log"

touch "$DONE"
echo "Done in $((SECONDS / 60)) min."

# Check the U2AF1 fix: the false copy (chr21 6.5 Mb) must be all N, and the
# real gene (chr21 43.09 Mb) must have bases.
echo "False copy: $(samtools faidx "$REF" chr21:6500001-6500060 | tail -n +2 | tr -d '\n')"
echo "Real U2AF1: $(samtools faidx "$REF" chr21:43094501-43094560 | tail -n +2 | tr -d '\n')"

Reference and indexes already in /content/my_project/reference/genome: skipped.


## STEP 1: Clean design files

In [25]:
%%bash
set -euo pipefail

# Inputs this step needs. [ -s FILE ] is true when FILE exists and isn't empty.
for f in Regions_SNV Covered_SNV; do
  [ -s "$HAEM_DIR/Haem_1_${f}.bed" ] || {
    echo "Missing $HAEM_DIR/Haem_1_${f}.bed: put the Haem_1 folder in reference/sureselect/ (see Before the first run)."
    exit 1; }
done
[ -s "${REF%.fa}.dict" ] || { echo "No reference dictionary: run step 0 first."; exit 1; }

# Clean the two design files we need:
#   Regions_SNV = exon targets    -> where variants are called and scored
#   Covered_SNV = probe footprint -> where reads come from (step 3)
for f in Regions_SNV Covered_SNV; do
  # 1) Drop Agilent's 'track' and 'browser' header lines; most tools reject
  #    them. -v keeps lines that DON'T match; -E lets (a|b) mean either word.
  grep -v -E '^(track|browser)' "$HAEM_DIR/Haem_1_${f}.bed" |
  # 2) Tab-separated output (OFS), skipping chr21 rows that start below 7 Mb:
  #    the nine rows on the false U2AF1 copy, which is all N in the masked
  #    reference. Keep chromosome, start, end and gene name only.
  awk 'BEGIN{OFS="\t"} !($1=="chr21" && $2<7000000) {print $1,$2,$3,$4}' |
  # 3) Sort by chromosome, then numerically (-n) by start.
  sort -k1,1 -k2,2n > "$DESIGN_DIR/${f}.clean.bed"
done

# Picard takes "interval lists": the same regions plus a header copied from the
# reference's .dict (-SD), so contig names and order match the BAMs. The picard
# command accepts Java memory options such as -Xmx4g before the tool name.
picard "$JAVA_MEM" BedToIntervalList -I "$TARGETS_BED" -O "$TARGETS_IL" -SD "${REF%.fa}.dict"
picard "$JAVA_MEM" BedToIntervalList -I "$BAITS_BED" -O "$BAITS_IL" -SD "${REF%.fa}.dict"

# Check against the counts this design gives.
check() {   # check FILE EXPECTED_ROWS EXPECTED_BP
  local rows bp
  rows=$(wc -l < "$1")
  bp=$(awk '{s += $3 - $2} END {print s}' "$1")
  if [ "$rows" -eq "$2" ] && [ "$bp" -eq "$3" ]; then
    echo "OK        $(basename "$1"): $rows rows, $bp bp"
  else
    echo "MISMATCH  $(basename "$1"): $rows rows, $bp bp (expected $2 rows, $3 bp)"
    return 1
  fi
}
check "$TARGETS_BED" 2170 380700
check "$BAITS_BED" 2068 485357

OK        Regions_SNV.clean.bed: 2170 rows, 380700 bp
OK        Covered_SNV.clean.bed: 2068 rows, 485357 bp


00:36:26.434 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/micromamba/envs/bioinf_portfolio/share/picard-3.5.0-1/picard.jar!/com/intel/gkl/native/libgkl_compression.so
[Mon Nov 23 00:36:26 UTC 2026] BedToIntervalList --INPUT /content/my_project/reference/sureselect/Regions_SNV.clean.bed --SEQUENCE_DICTIONARY /content/my_project/reference/genome/hg38_masked.dict --OUTPUT /content/my_project/reference/sureselect/targets.interval_list --SORT true --UNIQUE false --DROP_MISSING_CONTIGS false --KEEP_LENGTH_ZERO_INTERVALS false --VERBOSITY INFO --QUIET false --VALIDATION_STRINGENCY STRICT --COMPRESSION_LEVEL 5 --MAX_RECORDS_IN_RAM 500000 --CREATE_INDEX false --CREATE_MD5_FILE false --help false --version false --showHidden false --USE_JDK_DEFLATER false --USE_JDK_INFLATER false
[Mon Nov 23 00:36:26 UTC 2026] Executing as root@960882c89fb7 on Linux 6.6.159+ amd64; OpenJDK 64-Bit Server VM 17.0.18-internal+0-adhoc.rattler.src; Deflater: Intel; Inflater: Intel;

## STEP 2: Build the truth-variant VCF

Turn the truth-set table into checked hg38 records, each with its target VAF. Two cells: a Python cell checks a fixed table of positions offline, then a bash cell writes and checks the VCF.
*   In: the truth-set table (fixed in the first cell), the cleaned targets, the reference.
*   Out: data/raw/truth/variants.tsv (working list), truth_set.tsv (the same rows with MANE transcript, HGVS and IDs, for the write-up), spike.vcf (BAMSurgeon's input) and variants.norm.tsv (the normalised list step 9 joins on).

The first cell holds the truth set as a fixed table of GRCh38 positions. Each was looked up on GeneBe on 20 November 2026, which confirmed its HGVS on the MANE Select transcript. An earlier version asked Ensembl's REST API at run time, but Ensembl blocked Colab's shared network address. A fixed table also means a rerun can never quietly change the truth set. "Why a fixed table, and where the positions go", at the end of this step, explains the lookup and what the later steps do with each position. Offline, the cell checks that every row passes three tests:
*   the position lies inside a panel target, which also catches one on the false U2AF1 copy;
*   the reference carries the REF bases there, and each duplication inserts a copy of the bases right after its anchor;
*   it is at least 300 bp from every other spiked variant.
It stops with a list of problems if any check fails. It needs no internet; samtools is found through section 7's PATH.

In [26]:
import os
import subprocess
from pathlib import Path

TRUTH_DIR = Path(os.environ["TRUTH_DIR"])
REF = os.environ["REF"]

# The truth set, as a fixed table (see "The truth set"). Each row is one GRCh38
# position written the VCF way: 1-based, and an indel starts at the base just
# before it (the anchor), shifted as far left as it can go. Every row was
# checked on GeneBe (genebe.net/variant/hg38/CHROM-POS-REF-ALT) on 20 Nov 2026,
# which reports the HGVS on the MANE Select transcript given here. The cell
# re-checks what it can offline: REF bases, targets, duplications, spacing.
# Two rows use MANE numbering that differs from the familiar names:
#   MYD88 "L265P" is p.Leu252Pro (c.755T>C) on NM_002468.5.
#   CALR type 1, often written c.1092_1143del, is c.1099_1150del: the same
#   52 bp deletion, described from its rightmost position as HGVS requires.
CALR_DEL52 = "GCAGAGGCTTAAGGAGGAGGAAGAAGACAAGAAACGCAAAGAGGAGGAGGAG"   # the deleted bases
TRUTH = [
    # name            VAF   chrom    position   REF                ALT      MANE Select       HGVS c.           HGVS p.               ID
    ("JAK2_V617F",     0.40, "chr9",  5073770,   "G",               "T",     "NM_004972.4",    "c.1849G>T",      "p.Val617Phe",        "rs77375493"),
    ("SF3B1_K700E",    0.40, "chr2",  197402110, "T",               "C",     "NM_012433.4",    "c.2098A>G",      "p.Lys700Glu",        "rs559063155"),
    ("TP53_R273H",     0.40, "chr17", 7673802,   "C",               "T",     "NM_000546.6",    "c.818G>A",       "p.Arg273His",        "rs28934576"),
    ("ASXL1_G646Wfs12", 0.40, "chr20", 32434638, "A",               "AG",    "NM_015338.6",    "c.1934dup",      "p.Gly646TrpfsTer12", "ClinVar:426927"),
    ("IDH1_R132H",     0.10, "chr2",  208248388, "C",               "T",     "NM_005896.4",    "c.395G>A",       "p.Arg132His",        "rs121913500"),
    ("SRSF2_P95H",     0.10, "chr17", 76736877,  "G",               "T",     "NM_001195427.2", "c.284C>A",       "p.Pro95His",         "rs751713049"),
    ("MYD88_L265P",    0.10, "chr3",  38141150,  "T",               "C",     "NM_002468.5",    "c.755T>C",       "p.Leu252Pro",        "rs387907272"),
    ("CALR_type1",     0.10, "chr19", 12943750,  "A" + CALR_DEL52,  "A",     "NM_004343.4",    "c.1099_1150del", "p.Leu367ThrfsTer46", "rs1555760738"),
    ("IDH2_R140Q",     0.05, "chr15", 90088702,  "C",               "T",     "NM_002168.4",    "c.419G>A",       "p.Arg140Gln",        "rs121913502"),
    ("BRAF_V600E",     0.05, "chr7",  140753336, "A",               "T",     "NM_004333.6",    "c.1799T>A",      "p.Val600Glu",        "rs113488022"),
    ("U2AF1_S34F",     0.05, "chr21", 43104346,  "G",               "A",     "NM_006758.3",    "c.101C>T",       "p.Ser34Phe",         "rs371769427"),
    ("NRAS_G12D",      0.02, "chr1",  114716126, "C",               "T",     "NM_002524.5",    "c.35G>A",        "p.Gly12Asp",         "rs121913237"),
    ("KRAS_G12D",      0.02, "chr12", 25245350,  "C",               "T",     "NM_004985.5",    "c.35G>A",        "p.Gly12Asp",         "rs121913529"),
    ("DNMT3A_R882H",   0.02, "chr2",  25234373,  "C",               "T",     "NM_022552.5",    "c.2645G>A",      "p.Arg882His",        "rs147001633"),
    ("RHOA_G17V",      0.02, "chr3",  49375540,  "C",               "A",     "NM_001664.4",    "c.50G>T",        "p.Gly17Val",         "rs11552761"),
    ("NPM1_typeA",     0.02, "chr5",  171410539, "C",               "CTCTG", "NM_002520.7",    "c.860_863dup",   "p.Trp288CysfsTer12", "rs587776806"),
    ("DDX41_D140Gfs2", 0.50, "chr5",  177515944, "T",               "TCATC", "NM_016222.4",    "c.415_418dup",   "p.Asp140GlyfsTer2",  "rs762890562"),
    ("TP53_P72R",      0.50, "chr17", 7676154,   "G",               "C",     "NM_000546.6",    "c.215C>G",       "p.Pro72Arg",         "rs1042522"),
    ("TET2_I1762V",    0.50, "chr4",  105275794, "A",               "G",     "NM_001127208.3", "c.5284A>G",      "p.Ile1762Val",       "rs2454206"),
    ("FLT3_T227M",     0.50, "chr13", 28050157,  "G",               "A",     "NM_004119.3",    "c.680C>T",       "p.Thr227Met",        "rs1933437"),
]


def ref_seq(chrom, start, end):
    """Bases chrom:start-end (1-based, inclusive) from the masked reference."""
    out = subprocess.run(["samtools", "faidx", REF, f"{chrom}:{start}-{end}"],
                         check=True, capture_output=True, text=True).stdout
    return "".join(out.splitlines()[1:]).upper()   # drop the >name line, join the rest


# Target regions, to confirm every position is on the panel. This also catches
# a position on the false U2AF1 copy, whose rows step 1 removed.
TARGETS = {}
for line in open(os.environ["TARGETS_BED"]):
    chrom, start, end = line.split("\t")[:3]
    TARGETS.setdefault(chrom, []).append((int(start), int(end)))


def in_targets(chrom, pos):
    """True if the 1-based position lies in a target (BED starts are 0-based)."""
    return any(start < pos <= end for start, end in TARGETS.get(chrom, []))


rows, problems = [], []
for name, vaf, chrom, pos, ref, alt, tx, hgvs_c, hgvs_p, vid in TRUTH:
    # 1) The reference must carry the REF bases at this position. A wrong
    #    position, or a typo in REF, fails here.
    found = ref_seq(chrom, pos, pos + len(ref) - 1)
    if found != ref:
        problems.append(f"{name}: reference has {found[:12]!r} at {chrom}:{pos}, table says {ref[:12]!r}")
    # 2) The position must be inside a panel target.
    if not in_targets(chrom, pos):
        problems.append(f"{name}: {chrom}:{pos} is not inside a target")
    # 3) Indels keep the anchor base. A duplication, shifted left, inserts a copy
    #    of the bases straight after the anchor, so those must match.
    if len(alt) > len(ref):
        inserted = alt[len(ref):]
        after = ref_seq(chrom, pos + 1, pos + len(inserted))
        if alt[0] != ref[0] or inserted != after:
            problems.append(f"{name}: inserts {inserted!r}, but the bases after the anchor are {after!r}")
    elif len(ref) > len(alt) and alt != ref[0]:
        problems.append(f"{name}: a deletion's ALT must be the anchor base alone")
    rows.append((chrom, pos, ref, alt, vaf, name, tx, hgvs_c, hgvs_p, vid))
    shown = f"{len(ref) - 1} bp deleted" if len(ref) > 12 else f"{ref}>{alt}"   # CALR's REF is too long to print
    print(f"{name:16} {chrom + ':' + str(pos):16} {shown:14} {tx}:{hgvs_c} ({hgvs_p})")

# The FLT3-ITD, built by hand: no single ITD is standard. Take 30 bp inside the
# FLT3 exon 14 target (chr13:28,034,082-28,034,214) and insert a copy straight
# after it. 30 bp = 10 codons, so it stays in frame, as real ITDs do. In VCF an
# insertion is written at the base before the new bases (the anchor): here the
# last base of the stretch, with ALT = anchor + the 30 bp.
dup = ref_seq("chr13", 28034101, 28034130)
if len(dup) != 30 or "N" in dup:
    problems.append(f"FLT3_ITD30: unexpected reference sequence {dup!r}")
else:
    rows.append(("chr13", 28034130, dup[-1], dup[-1] + dup, 0.05, "FLT3_ITD30",
                 "NM_004119.3", "30 bp ITD built from chr13:28034101-28034130", ".", "."))
    print(f"{'FLT3_ITD30':16} {'chr13:28034130':16} {'+30 bp':14} copy of chr13:28034101-28034130 (exon 14)")

# Spacing: keep spiked variants at least ~300 bp apart (one fragment length),
# so no read pair carries two of them.
rows.sort(key=lambda r: (r[0], r[1]))
for a, b in zip(rows, rows[1:]):
    if a[0] == b[0] and b[1] - a[1] < 300:
        problems.append(f"{a[5]} and {b[5]} are only {b[1] - a[1]} bp apart (need >= 300)")

if problems or len(rows) != 21:
    raise RuntimeError(f"{len(rows)} rows, {len(problems)} problem(s):\n  " + "\n  ".join(problems))

# Working list for step 2's bash cell: CHROM POS REF ALT VAF NAME, no header.
with open(TRUTH_DIR / "variants.tsv", "w") as out:
    for r in rows:
        out.write("\t".join(str(x) for x in r[:6]) + "\n")
# The full table with its annotation, for the README and the results write-up.
with open(TRUTH_DIR / "truth_set.tsv", "w") as out:
    out.write("chrom\tpos\tref\talt\tvaf\tname\tmane_select\thgvs_c\thgvs_p\tid\n")
    for r in rows:
        out.write("\t".join(str(x) for x in r) + "\n")
print(f"\nAll 21 variants passed the checks.")
print(f"Wrote {TRUTH_DIR / 'variants.tsv'} and {TRUTH_DIR / 'truth_set.tsv'}")

JAK2_V617F       chr9:5073770     G>T            NM_004972.4:c.1849G>T (p.Val617Phe)
SF3B1_K700E      chr2:197402110   T>C            NM_012433.4:c.2098A>G (p.Lys700Glu)
TP53_R273H       chr17:7673802    C>T            NM_000546.6:c.818G>A (p.Arg273His)
ASXL1_G646Wfs12  chr20:32434638   A>AG           NM_015338.6:c.1934dup (p.Gly646TrpfsTer12)
IDH1_R132H       chr2:208248388   C>T            NM_005896.4:c.395G>A (p.Arg132His)
SRSF2_P95H       chr17:76736877   G>T            NM_001195427.2:c.284C>A (p.Pro95His)
MYD88_L265P      chr3:38141150    T>C            NM_002468.5:c.755T>C (p.Leu252Pro)
CALR_type1       chr19:12943750   52 bp deleted  NM_004343.4:c.1099_1150del (p.Leu367ThrfsTer46)
IDH2_R140Q       chr15:90088702   C>T            NM_002168.4:c.419G>A (p.Arg140Gln)
BRAF_V600E       chr7:140753336   A>T            NM_004333.6:c.1799T>A (p.Val600Glu)
U2AF1_S34F       chr21:43104346   G>A            NM_006758.3:c.101C>T (p.Ser34Phe)
NRAS_G12D        chr1:114716126   C>T            NM

Write the VCF and check every REF base against the reference:

In [27]:
%%bash
set -euo pipefail

[ -s "$TRUTH_DIR/variants.tsv" ] || { echo "No variants.tsv: run the cell above first."; exit 1; }

# Build a minimal VCF: header lines, then one line per variant.
{
  printf '##fileformat=VCFv4.2\n'
  # The two INFO fields: BAMSurgeon reads VAF as the fraction to spike; NAME
  # is only a label for us.
  printf '##INFO=<ID=VAF,Number=1,Type=Float,Description="Target allele fraction">\n'
  printf '##INFO=<ID=NAME,Number=1,Type=String,Description="Variant name">\n'
  # One ##contig line per reference sequence, so bcftools knows the order.
  awk '{printf "##contig=<ID=%s,length=%s>\n", $1, $2}' "$GENOME"
  printf '#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n'
  # ID and QUAL "." (empty), FILTER PASS, INFO from columns 5 and 6.
  awk 'BEGIN{OFS="\t"} {print $1,$2,".",$3,$4,".","PASS","VAF="$5";NAME="$6}' "$TRUTH_DIR/variants.tsv"
} > "$TRUTH_DIR/spike.unsorted.vcf"

# Sort into reference order (-Ov = plain-text VCF).
bcftools sort "$TRUTH_DIR/spike.unsorted.vcf" -Ov -o "$TRUTH_DIR/spike.sorted.vcf"

# Normalise, which also catches mistakes:
#   -f REF         compare against the reference
#   --check-ref e  stop with an error if any REF base doesn't match it
#   -m -both       split any multi-allelic record into one per ALT
#   (always)       left-align indels, so each has one standard position
bcftools norm -f "$REF" --check-ref e -m -both "$TRUTH_DIR/spike.sorted.vcf" -Ov -o "$TRUTH_DIR/spike.vcf"

# The same list as spike.vcf after normalisation. Step 9 joins results on
# these positions, which is what the truth VCF will carry.
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%INFO/VAF\t%INFO/NAME\n' "$TRUTH_DIR/spike.vcf" \
  > "$TRUTH_DIR/variants.norm.tsv"

echo "Records in spike.vcf: $(grep -vc '^#' "$TRUTH_DIR/spike.vcf") (expect 21)"

Records in spike.vcf: 21 (expect 21)


Writing to /tmp/bcftools.leHddK
Merging 1 temporary files
Done
Cleaning
Lines   total/split/joined/realigned/mismatch_removed/dup_removed/skipped:	21/0/0/1/0/0/0


Check: 21 records, and bcftools norm finished without a REF-mismatch error. Its summary line shows realigned 1: the FLT3-ITD, which step 2 anchors at the right end of the duplicated stretch and bcftools shifts to the left end, the standard VCF form. The other indels were already left-aligned (seen on the first Colab run).

What the first run showed (20 Nov 2026). All 21 rows passed the Python cell's checks, spike.vcf has 21 records, and bcftools norm realigned exactly one. Three results show that the table and your reference agree:

*   CALR type 1: all 53 REF bases matched. The row's REF is the anchor base plus the 52 deleted bases, and the cell compares every one with the reference. At a wrong position, or with the deletion written from the wrong end, the bases would stop matching within the first few, and 53 in a row don't match by chance. It also confirms the two ways of writing this deletion describe the same 52 bases: VCF from its left end (chr19:12,943,750), HGVS from its right end (c.1099_1150del).
*  U2AF1 S34F: the REF base G was found at chr21:43,104,346. That is the real U2AF1 gene, which the masked reference keeps; only the false copy near chr21:6.5 Mb is turned to N (step 0's printout showed both). Had the masking hit the real gene, the cell would have read N there and stopped. So reads from U2AF1 have one place to map, and this variant can be spiked and called.
*   FLT3-ITD: moved from chr13:28,034,130 to 28,034,100, exactly 30 bases. Step 2 writes the inserted copy at the right end of the duplicated stretch, but VCF places an insertion at the leftmost spot it could occupy. An exact tandem copy can slide back by its own length, so the anchor moved to the base just before the stretch: 28,034,130 − 30 = 28,034,100. It stopped there because that base differs from the stretch's last base, so the repeat goes no further. A shift of exactly the duplication's length is what a correctly built tandem duplication should do, and the new position is still inside the exon 14 target (28,034,082–28,034,214). Check it with grep FLT3_ITD30 "$TRUTH_DIR/variants.norm.tsv", which shows chr13  28034100. Step 9 joins on this normalised position, which is the one BAMSurgeon's truth file will carry.

The other three indels (ASXL1, NPM1, DDX41) weren't moved, so GeneBe's positions were already left-aligned. Their duplication check also passed: each inserts a copy of the bases right after its anchor, as a duplication should.

### Why a fixed table, and where the positions go

What has to be translated. The truth-set table describes each variant the way a clinical report does: a gene, and an HGVS change on one of its transcripts, such as JAK2 c.1849G>T (p.Val617Phe). Every tool after step 2 works in genome coordinates instead: chromosome, position, reference base(s) and alternative base(s). So something has to turn "c.1849G>T in JAK2" into "chr9, position 5,073,770, G becomes T".

Why the transcript matters. c.1849 means the 1,849th base of the coding sequence, counted from the A of the start codon along one particular transcript, with the introns skipped. A gene's transcripts can differ in their exons or start codon, so the same c. number can sit on different genomic bases. MYD88 shows this: the base called c.794 on the older transcript NM_002468.4 is c.755 on the MANE transcript NM_002468.5, which has 39 fewer coding bases before it. Clinical reports use MANE Select, so the table does too.

What the Ensembl version did, for each row (JAK2 V617F as the example):
1.  Find the transcript. Ask Ensembl's lookup endpoint for JAK2's transcripts and keep the one flagged canonical. For genes with a MANE Select transcript, Ensembl makes that the canonical one: here ENST00000381652, which matches RefSeq's NM_004972.4.
2.  Build the full HGVS. Join the two: ENST00000381652:c.1849G>T.
3.  Convert it. Send that to Ensembl's Variant Recoder. Ensembl stores where each exon of the transcript lies on GRCh38, so it can walk along the coding exons to base 1,849 and report the genomic position in VCF form, 9-5073770-G-T, plus the protein change it causes, p.Val617Phe. For a gene on the minus strand the bases come back complemented: TP53 c.818G>A is C>T on the genome.
4.  Check it. Keep the position only if it lies inside a panel target, the protein change matches the table, and it is at least 300 bp from every other variant.
5.  Write it down. Add one line to variants.tsv: chr9  5073770  G  T  0.40  JAK2_V617F.
The transcript is used only in steps 1–3. From step 4 on, every file holds just chromosome, position, REF, ALT, VAF and name.
What the fixed table changes. Steps 1–3 were done once, by hand, on GeneBe, which shows the HGVS on the MANE Select transcript for any GRCh38 position, and their results are typed into the cell. The cell starts at step 4, with one difference: without the transcript's exon map it can't work out the protein change, so it checks the REF bases against the reference instead, alongside the target, duplication and spacing checks. Step 5 is unchanged, so every later step gets exactly the same file either way.
Where the JAK2 position goes next.
1.  Step 2, bash cell. variants.tsv becomes a VCF record in spike.vcf: chr9  5073770  .  G  T  .  PASS  VAF=0.4;NAME=JAK2_V617F. bcftools norm confirms the reference really has a G at chr9:5,073,770 and shifts any indel to its leftmost position. The normalised list is saved as variants.norm.tsv.
2.  Step 6, BAMSurgeon. It reads that record and collects Sample A's reads covering chr9:5,073,770 (roughly 900). It picks 40% of them at random, changes the G to T in those reads and realigns them. The result is Sample B. It also writes truth.vcf.gz, recording what actually went in and the VAF it achieved.
3.  Step 7. Sample B is turned back into FASTQ, so the test pipeline starts from raw reads, as it would with a real sample.
4.  Step 8, the pipeline under test. It aligns the reads and calls variants with Mutect2, without ever seeing the truth set. If it works, B.pass.vcf.gz holds a call at chr9:5,073,770 G>T with an allele fraction near 0.4.
5.  Step 9, scoring. vcfeval lines up truth.vcf.gz against the pipeline's calls. A truth variant with a matching call is a true positive, one with no call is a false negative, and a call that matches nothing is a false positive. The R table then joins vcfeval's verdicts to variants.norm.tsv on chromosome and position, so each row gets its name and tier back: "JAK2_V617F, 40%, found". Sample A had nothing spiked in, so any call inside its targets counts as a false positive.
So the positions are the answer key. They tell BAMSurgeon where to write each variant, and they tell vcfeval where the pipeline should find it.
Why the table is fixed rather than looked up at run time.
*   Ensembl's REST service refused Colab's shared network address on 20 November 2026, so the lookup couldn't run there.
*   A truth set should be a fixed, versioned file. A live lookup follows Ensembl's current release, so a rerun could quietly pick a different transcript.
*   The live version needed each HGVS to be exactly right on the current MANE transcript. The plan's MYD88 row, c.794T>C, would have failed its protein check.
*   What's lost: the protein check was done once, on GeneBe (see Sources), rather than on every run.